In [1]:
%load_ext lab_black
%load_ext autoreload
%autoreload 2

import pandas as pd
import numpy as np
import polars as pl
import time
from tqdm.notebook import tqdm
from sklearn.model_selection import train_test_split
from sklearn.metrics import roc_auc_score, accuracy_score
from sklearn.preprocessing import OrdinalEncoder
import os
import gc
import matplotlib.pyplot as plt
import warnings
import sys
import math

import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.utils.data import Dataset
import random

os.environ["CUDA_VISIBLE_DEVICES"] = "-1"
from merlin.loader.torch import Loader
from merlin.io import Dataset as Dataset_merlin

os.environ["CUDA_VISIBLE_DEVICES"] = "0"

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
# device = torch.device("cpu")
print(device)

sys.path.append("../src/otto_recommender")
import utils

warnings.filterwarnings("ignore")
tqdm.pandas()

pd.set_option("max_rows", 500)

%matplotlib inline

cuda


In [2]:
def seed_everything(seed):
    random.seed(seed)
    os.environ["PYTHONHASHSEED"] = str(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False  # set True to be faster
    # torch.use_deterministic_algorithms(True)
    print(f"Setting all seeds to be {seed} to reproduce...")


seed_everything(42)

Setting all seeds to be 42 to reproduce...


In [3]:
class Config:
    col_user = "user"
    col_aid = "aid"
    col_type = "type"
    col_n_session = "n_session"
    col_n_day = "n_day"
    col_n_week = "n_week"
    col_dayofweek = "dayofweek"
    col_hour = "hour"
    col_ts = "ts"
    col_num_action = "num_action"
    col_train_seq = "train_seq"
    col_target = "target"
    col_type_group = "type_group"
    batch_size_train = 256
    batch_size_infer = 256
    n_epochs = 10
    max_len = 10
    model_name = "target_action"
    hidden_dim = 32
    embedding_dim = 64


config = Config()

In [4]:
print(utils.recall_vals_one_row([1, 2, 3], 2))
print(utils.recall_vals_one_row([1, 2, 3], 4))

True
False


In [5]:
utils.recall_several_targets(candidates=[[1, 2, 3], [1]], targets=[[1, 3, 2], [1, 2]])

0.8

In [6]:
from pandarallel import pandarallel

pandarallel.initialize(nb_workers=12, progress_bar=True, use_memory_fs=False)

INFO: Pandarallel will run on 12 workers.
INFO: Pandarallel will use standard multiprocessing data transfer (pipe) to transfer data between the main process and workers.


In [7]:
%%time

PATH_DATA = '../data/02_intermediate/'
df_train = pl.read_parquet(PATH_DATA + 'df_train_n_session.parquet')
df_test = pl.read_parquet(PATH_DATA + 'df_test_n_session.parquet')
# df_train = df_train.filter(pl.col("user") % 100 == 0)
df_train = df_train.to_pandas()
df_test = df_test.to_pandas()
# df_test = catalog.load("df_test")

df_train.name = "df_train"
df_test.name = "df_test"

df_train = utils.make_date_stats(df_train)
df_test = utils.make_date_stats(df_test)

df_train.drop(['date_day'], axis=1, inplace=True)
df_test.drop(['date_day'], axis=1, inplace=True)

df_train = pl.from_pandas(df_train)
df_test = pl.from_pandas(df_test)

CPU times: user 1min 53s, sys: 28.4 s, total: 2min 22s
Wall time: 2min 6s


In [8]:
df_train.select(pl.col("n_week").unique())

n_week
i8
31
32
33
34


In [9]:
%%time

users_with_carts_or_orders_train = (
    df_train.filter(
        (pl.col(config.col_type) == "carts") | (pl.col(config.col_type) == "orders")
    )
    .select(config.col_user)
    .unique()
)

users_with_carts_or_orders_test = (
    df_test.filter(
        (pl.col(config.col_type) == "carts") | (pl.col(config.col_type) == "orders")
    )
    .select(config.col_user)
    .unique()
)

df_train = df_train.filter(
    pl.col(config.col_user).is_in(
        list(users_with_carts_or_orders_train.to_pandas()[config.col_user])
    )
)

df_test = df_test.filter(
    pl.col(config.col_user).is_in(
        list(users_with_carts_or_orders_test.to_pandas()[config.col_user])
    )
)

CPU times: user 37.7 s, sys: 25.1 s, total: 1min 2s
Wall time: 9.39 s


In [10]:
%%time

df_train = df_train.with_columns(
    [(pl.col(config.col_user) + "_" + pl.col(config.col_n_week)).alias("user_week")]
)

df_test = df_test.with_columns(
    [(pl.col(config.col_user) + "_" + pl.col(config.col_n_week)).alias("user_week")]
)

users_weeks_with_carts_or_orders_train = (
    df_train.filter(
        (pl.col(config.col_type) == "carts") | (pl.col(config.col_type) == "orders")
    )
    .select("user_week")
    .unique()
)

users_weeks_with_carts_or_orders_test = (
    df_test.filter(
        (pl.col(config.col_type) == "carts") | (pl.col(config.col_type) == "orders")
    )
    .select("user_week")
    .unique()
)

df_train = df_train.filter(
    pl.col("user_week").is_in(
        list(users_weeks_with_carts_or_orders_train.to_pandas()["user_week"])
    )
)

df_test = df_test.filter(
    pl.col("user_week").is_in(
        list(users_weeks_with_carts_or_orders_test.to_pandas()["user_week"])
    )
)

CPU times: user 1min 8s, sys: 55.5 s, total: 2min 3s
Wall time: 28 s


In [11]:
%%time

test_users = df_test[config.col_user].unique()
df = pl.concat([df_train, df_test])
df = df.sort(config.col_ts, reverse=True)

df = df.to_pandas()
df[config.col_type_group] = df[config.col_type].apply(lambda x: 'clicks' if x == 'clicks' else 'target_action')
df[config.col_num_action + '_per_type_group'] = df.groupby([config.col_user, config.col_type_group])[config.col_aid].cumcount()
df[config.col_num_action] = df.groupby([config.col_user])[config.col_aid].cumcount()
df = pl.from_pandas(df)

df = df.with_columns(
    [
        pl.col(config.col_num_action)
        .cast(pl.Int32, strict=True)
        .alias(config.col_num_action),
        pl.col(config.col_n_session)
        .cast(pl.Int32, strict=True)
        .alias(config.col_n_session),
    ]
)

CPU times: user 5min 6s, sys: 1min 2s, total: 6min 8s
Wall time: 1min 47s


In [12]:
# ### IF DEBUG PIPELINE !!!

# df = df.filter(pl.col("user") % 100 == 0)

In [13]:
df

user,aid,ts,type,n_session,date,n_day,n_week,dayofweek,hour,user_week,type_group,num_action_per_type_group,num_action
i64,i64,i64,str,i32,datetime[ns],i8,i8,i8,i8,str,str,i64,i32
13629931,1576043,1662328770401,"""clicks""",1,2022-09-04 23:59:30.400999936,4,35,6,23,"""13629931_35""","""clicks""",0,0
14568911,231093,1662328768077,"""clicks""",0,2022-09-04 23:59:28.076999936,4,35,6,23,"""14568911_35""","""clicks""",0,0
14250687,374291,1662328764970,"""carts""",2,2022-09-04 23:59:24.969999872,4,35,6,23,"""14250687_35""","""target_action""",0,0
14362537,1142000,1662328764079,"""clicks""",1,2022-09-04 23:59:24.079000064,4,35,6,23,"""14362537_35""","""clicks""",0,0
13723496,1240764,1662328761140,"""carts""",1,2022-09-04 23:59:21.140,4,35,6,23,"""13723496_35""","""target_action""",0,0
14570806,499919,1662328761104,"""clicks""",0,2022-09-04 23:59:21.104,4,35,6,23,"""14570806_35""","""clicks""",0,0
14569514,36824,1662328760228,"""clicks""",0,2022-09-04 23:59:20.228,4,35,6,23,"""14569514_35""","""clicks""",0,0
14025789,148572,1662328760126,"""clicks""",1,2022-09-04 23:59:20.126000128,4,35,6,23,"""14025789_35""","""clicks""",0,0
14571190,1685915,1662328755424,"""clicks""",0,2022-09-04 23:59:15.424,4,35,6,23,"""14571190_35""","""clicks""",0,0


In [14]:
del df_train, df_test
gc.collect()

2830

In [15]:
%%time

counts_aid = df.groupby([config.col_aid]).count().sort("count", reverse=True)

trs = 40
counts_aid_filtered = counts_aid.filter(pl.col("count") >= trs).to_pandas()
pad = len(counts_aid_filtered)

ordinal_encoder = OrdinalEncoder(handle_unknown="use_encoded_value", unknown_value=pad)
ordinal_encoder.fit(counts_aid_filtered[config.col_aid].values.reshape(-1, 1))

CPU times: user 51.6 s, sys: 24.1 s, total: 1min 15s
Wall time: 3.59 s


OrdinalEncoder(handle_unknown='use_encoded_value', unknown_value=413313)

In [16]:
%%time

X_train, X_val = train_test_split(df, test_size=0.1, random_state=42)

CPU times: user 6min 59s, sys: 1min 8s, total: 8min 8s
Wall time: 20.9 s


In [17]:
%%time

X_train = X_train.to_pandas()

X_train[config.col_aid] = ordinal_encoder.transform(
    X_train[config.col_aid].values.reshape(-1, 1)
).astype(int)
X_train = X_train[X_train[config.col_aid] != pad]

X_train = pl.from_pandas(X_train)

CPU times: user 1min 46s, sys: 15.6 s, total: 2min 2s
Wall time: 1min 57s


In [18]:
del df
gc.collect()

190

In [19]:
%%time

def make_target_action_cols(df):
    df = df.with_columns(
        [
            pl.when(
                # (pl.col(config.col_num_action + "_per_type_group") == 0) &
                 (pl.col(config.col_type_group) == "target_action")
            )
            .then(1)
            .otherwise(0)
            .alias("target_action_done")
        ]
    )
    
    df = df.to_pandas()
    df["target_action_done_cumsum"] = df.groupby([config.col_user])["target_action_done"].cumsum()
    df = pl.from_pandas(df)
    df = df.sort(config.col_ts, reverse=False)
    df = df.to_pandas()
    df["target_action_done_cumsum_per_week_reversed"] = df.groupby([config.col_user, config.col_n_week])["target_action_done"].cumsum()
    df = df[df["target_action_done_cumsum"] > 0]
    df = pl.from_pandas(df)
    # df = df.sort(config.col_ts, reverse=True)
    df = df.with_columns(
        [
            pl.col('target_action_done')
            .cast(pl.Int16, strict=True)
            .alias('target_action_done'),
            pl.col("target_action_done_cumsum")
            .cast(pl.Int16, strict=True)
            .alias("target_action_done_cumsum"),
            pl.col("target_action_done_cumsum_per_week_reversed")
            .cast(pl.Int16, strict=True)
            .alias("target_action_done_cumsum_per_week_reversed"),
        ]
    )    
    return df

X_train = make_target_action_cols(X_train)
X_val = make_target_action_cols(X_val)

CPU times: user 9min 22s, sys: 1min 49s, total: 11min 12s
Wall time: 2min 45s


In [20]:
X_train.filter(pl.col("user_week") == "0_34")

user,aid,ts,type,n_session,date,n_day,n_week,dayofweek,hour,user_week,type_group,num_action_per_type_group,num_action,target_action_done,target_action_done_cumsum,target_action_done_cumsum_per_week_reversed
i64,i64,i64,str,i32,datetime[ns],i8,i8,i8,i8,str,str,i64,i32,i16,i16,i16
0,2304,1661157951797,"""clicks""",32,2022-08-22 10:45:51.796999936,22,34,0,10,"""0_34""","""clicks""",112,128,0,15,0
0,253778,1661157993281,"""clicks""",32,2022-08-22 10:46:33.280999936,22,34,0,10,"""0_34""","""clicks""",110,126,0,4,0
0,203470,1661158034888,"""clicks""",32,2022-08-22 10:47:14.888,22,34,0,10,"""0_34""","""clicks""",109,125,0,15,0
0,106586,1661158090652,"""clicks""",32,2022-08-22 10:48:10.652,22,34,0,10,"""0_34""","""clicks""",108,124,0,19,0
0,114587,1661158253105,"""clicks""",32,2022-08-22 10:50:53.104999936,22,34,0,10,"""0_34""","""clicks""",105,121,0,19,0
0,321676,1661206127572,"""clicks""",33,2022-08-23 00:08:47.572,23,34,1,0,"""0_34""","""clicks""",103,119,0,15,0
0,148154,1661240281936,"""clicks""",34,2022-08-23 09:38:01.936,23,34,1,9,"""0_34""","""clicks""",100,116,0,15,0
0,401366,1661240292181,"""clicks""",34,2022-08-23 09:38:12.180999936,23,34,1,9,"""0_34""","""clicks""",99,115,0,15,0
0,404019,1661289160324,"""clicks""",35,2022-08-23 23:12:40.324,23,34,1,23,"""0_34""","""clicks""",96,112,0,11,0


In [21]:
%%time

X_train = X_train.groupby([config.col_user, config.col_n_week, "user_week"]).agg(
    [
        pl.col(config.col_aid).list().alias("seq_" + config.col_aid),
        pl.col("target_action_done_cumsum_per_week_reversed")
        .list()
        .alias("seq_" + "target_action_done_cumsum_per_week_reversed"),
        pl.col("type_group").list().alias("seq_" + "type_group"),
    ]
).filter(pl.col("seq_aid").apply(lambda x: len(x)) >= 2)

CPU times: user 2min 18s, sys: 9.97 s, total: 2min 28s
Wall time: 40.7 s


In [22]:
%%time

X_val = X_val.groupby([config.col_user, config.col_n_week, "user_week"]).agg(
    [
        pl.col(config.col_aid).list().alias("seq_" + config.col_aid),
        pl.col("target_action_done_cumsum_per_week_reversed")
        .list()
        .alias("seq_" + "target_action_done_cumsum_per_week_reversed"),
        pl.col("type_group").list().alias("seq_" + "type_group"),
    ]
).filter(pl.col("seq_aid").apply(lambda x: len(x)) >= 2)

CPU times: user 16.1 s, sys: 1.28 s, total: 17.4 s
Wall time: 7.75 s


In [23]:
X_train.head()

user,n_week,user_week,seq_aid,seq_target_action_done_cumsum_per_week_reversed,seq_type_group
i64,i8,str,list[i64],list[i16],list[str]
411150,31,"""411150_31""","[65500, 108855, ... 197625]","[0, 0, ... 31]","[""clicks"", ""clicks"", ... ""clicks""]"
486149,31,"""486149_31""","[108030, 395135, ... 324144]","[0, 0, ... 1]","[""clicks"", ""clicks"", ... ""clicks""]"
4068032,31,"""4068032_31""","[400952, 247774, ... 116718]","[0, 0, ... 1]","[""clicks"", ""clicks"", ... ""target_action""]"
4645344,31,"""4645344_31""","[145094, 145094, ... 351753]","[0, 1, ... 2]","[""clicks"", ""target_action"", ... ""clicks""]"
4774309,31,"""4774309_31""","[187597, 187597, ... 405201]","[0, 1, ... 2]","[""clicks"", ""target_action"", ... ""clicks""]"


In [24]:
X_train.filter(pl.col(config.col_user) == 1670369)

user,n_week,user_week,seq_aid,seq_target_action_done_cumsum_per_week_reversed,seq_type_group
i64,i8,str,list[i64],list[i16],list[str]
1670369,33,"""1670369_33""","[209946, 200588, ... 219649]","[0, 0, ... 6]","[""clicks"", ""clicks"", ... ""clicks""]"
1670369,32,"""1670369_32""","[52500, 265246, ... 12864]","[0, 0, ... 19]","[""clicks"", ""clicks"", ... ""clicks""]"


In [25]:
df_ex = X_train.head()
df_ex

user,n_week,user_week,seq_aid,seq_target_action_done_cumsum_per_week_reversed,seq_type_group
i64,i8,str,list[i64],list[i16],list[str]
411150,31,"""411150_31""","[65500, 108855, ... 197625]","[0, 0, ... 31]","[""clicks"", ""clicks"", ... ""clicks""]"
486149,31,"""486149_31""","[108030, 395135, ... 324144]","[0, 0, ... 1]","[""clicks"", ""clicks"", ... ""clicks""]"
4068032,31,"""4068032_31""","[400952, 247774, ... 116718]","[0, 0, ... 1]","[""clicks"", ""clicks"", ... ""target_action""]"
4645344,31,"""4645344_31""","[145094, 145094, ... 351753]","[0, 1, ... 2]","[""clicks"", ""target_action"", ... ""clicks""]"
4774309,31,"""4774309_31""","[187597, 187597, ... 405201]","[0, 1, ... 2]","[""clicks"", ""target_action"", ... ""clicks""]"


In [26]:
df_ex = df_ex.to_pandas()

In [27]:
n_iter = 5
mod = "train"

for i in tqdm(range(len(df_ex))):
    row = df_ex.iloc[i]
    row_seq_target_action_done_cumsum_per_week_reversed = row[
        "seq_target_action_done_cumsum_per_week_reversed"
    ]
    if len(set(row_seq_target_action_done_cumsum_per_week_reversed)) > 1:
        row_seq_aid = row["seq_aid"]
        row_seq_type_group = row["seq_type_group"]
        len_row_train = len(row_seq_aid)
        n_iter_cur = min(n_iter, len_row_train - 1)
        idxs_target_action = np.where(row_seq_type_group == "target_action")[0]
        for idx in idxs_target_action:
            train_seq = row_seq_aid[:idx]
            target_seq = row_seq_aid[idx:]

  0%|          | 0/5 [00:00<?, ?it/s]

In [28]:
idxs_target_action

array([1, 2])

In [29]:
row_seq_aid

array([187597, 187597, 279116, 187597, 151595,  25529, 290321, 306018,
       411191, 306018, 177290, 332585, 103640, 405201])

In [30]:
row_seq_aid[:6]

array([187597, 187597, 279116, 187597, 151595,  25529])

In [31]:
row_seq_aid[6:]

array([290321, 306018, 411191, 306018, 177290, 332585, 103640, 405201])

In [32]:
row_seq_type_group

array(['clicks', 'target_action', 'target_action', 'clicks', 'clicks',
       'clicks', 'clicks', 'clicks', 'clicks', 'clicks', 'clicks',
       'clicks', 'clicks', 'clicks'], dtype=object)

In [33]:
def make_seqs(df, pad=pad, mod="train"):

    df = df.to_pandas()

    list_train = list()
    list_target = list()
    list_user = list()
    list_user_week = list()

    for i in tqdm(range(len(df))):
        row = df.iloc[i]
        row_seq_target_action_done_cumsum_per_week_reversed = row[
            "seq_target_action_done_cumsum_per_week_reversed"
        ]
        if len(set(row_seq_target_action_done_cumsum_per_week_reversed)) > 1:
            row_seq_aid = row["seq_aid"]
            row_seq_type_group = row["seq_type_group"]
            row_user = row[config.col_user]
            row_user_week = row[config.col_user + "_week"]
            len_row_train = len(row_seq_aid)
            n_iter_cur = min(n_iter, len_row_train - 1)
            idxs_target_action = np.where(row_seq_type_group == "target_action")[0]
            if mod == "train":
                for idx in range(len(idxs_target_action)):
                    idx_chosen = idxs_target_action[idx]
                    idx_target = idxs_target_action[idx:]
                    train_seq = row_seq_aid[:idx_chosen].tolist()
                    target_seq = tuple(set(row_seq_aid[idx_target]))
                    if len(train_seq) > 0:
                        train_seq = tuple(
                            utils.truncate_list(
                                l=train_seq, max_len=config.max_len, from_start=False
                            )
                        )
                        for target in target_seq:
                            list_train.append(train_seq)
                            list_target.append(target)
                            list_user.append(row_user)
                            list_user_week.append(row_user_week)
            elif mod == "val":
                random.seed(i)
                idx = random.randint(0, len(idxs_target_action) - 1)
                idx_chosen = idxs_target_action[idx]
                idx_target = idxs_target_action[idx:]
                train_seq = row_seq_aid[:idx_chosen].tolist()
                target_seq = list(set(row_seq_aid[idx_target]))
                if len(train_seq) > 0:
                    train_seq = tuple(train_seq)
                    target_seq = tuple(
                        utils.truncate_list(l=target_seq, max_len=20, from_start=True)
                    )
                    list_train.append(train_seq)
                    list_target.append(target_seq)
                    list_user.append(row_user)
                    list_user_week.append(row_user_week)

    df = pd.DataFrame()
    df[config.col_user] = list_user
    df[config.col_user + "_week"] = list_user_week
    df[config.col_train_seq] = list_train
    df[config.col_target] = list_target
    df.drop_duplicates(inplace=True)

    if mod == "val":
        df = df.explode(config.col_train_seq)
        df[config.col_train_seq] = ordinal_encoder.transform(
            df[config.col_train_seq].values.reshape(-1, 1)
        ).astype(int)
        df = df[df[config.col_train_seq] != pad]
        df = (
            df.groupby([config.col_user, config.col_user + "_week", config.col_target])[
                [config.col_train_seq]
            ]
            .agg(list)
            .reset_index()
        )
        df[config.col_train_seq] = df[config.col_train_seq].apply(
            lambda x: utils.truncate_list(
                l=list(x), max_len=config.max_len, from_start=False
            )
        )
        df = df[
            [
                config.col_user,
                config.col_user + "_week",
                config.col_train_seq,
                config.col_target,
            ]
        ]

    df = pl.from_pandas(df)
    # df = df.unique(
    #     subset=[
    #         config.col_user,
    #         config.col_user + "_week",
    #         config.col_train_seq,
    #         config.col_target,
    #     ]
    # )
    return df

In [34]:
# X_test.head(10)

In [35]:
# %%time

# test = make_seqs(X_test.head(10), mod="val")

In [36]:
# test

In [37]:
# X_train.head(10)

In [38]:
# X_train.filter(pl.col("user_week") == "4206123_31")

In [39]:
# list(X_train.filter(pl.col("user_week") == "4206123_31")["seq_aid"])

In [40]:
# list(X_train.filter(pl.col("user_week") == "4206123_31")["seq_type_group"])

In [41]:
# %%time

# train = make_seqs(X_train.head(10), mod="train")

In [42]:
# train

In [43]:
%%time

X_val = make_seqs(X_val, mod="val")

  0%|          | 0/1198493 [00:00<?, ?it/s]

CPU times: user 2min 55s, sys: 2.93 s, total: 2min 57s
Wall time: 2min 57s


In [44]:
%%time

X_train = make_seqs(X_train, mod="train")

  0%|          | 0/4722483 [00:00<?, ?it/s]

CPU times: user 15min 15s, sys: 40.1 s, total: 15min 55s
Wall time: 15min 53s


In [45]:
X_train.head()

user,user_week,train_seq,target
i64,str,list[i64],i64
411150,"""411150_31""","[65500, 108855, 45847]",221344
411150,"""411150_31""","[65500, 108855, 45847]",115009
411150,"""411150_31""","[65500, 108855, 45847]",207548
411150,"""411150_31""","[65500, 108855, 45847]",194309
411150,"""411150_31""","[65500, 108855, 45847]",143399


In [46]:
X_val.head()

user,user_week,train_seq,target
i64,str,list[i64],list[i64]
0,"""0_34""",[397718],[1521766]
3,"""3_31""","[215323, 226403, ... 345815]",[128326]
14,"""14_31""","[96583, 277598, ... 348555]","[284816, 125127]"
15,"""15_31""","[140781, 78558]",[1409430]
19,"""19_31""","[272778, 104886, ... 85086]",[383546]


In [47]:
X_train.shape

(67627159, 4)

In [48]:
X_val.shape

(694051, 4)

In [49]:
# def split_data(df, validation_cut, test_users):
#     df_val = df[
#         (df[config.col_num_action] <= validation_cut)
#         & (df[config.col_user].isin(test_users))
#     ]

#     df_val_seq = df[
#         (df[config.col_num_action] > validation_cut)
#         & (df[config.col_user].isin(test_users))
#     ]

#     df_train = df.drop(df_val.index)
#     return df_train, df_val, df_val_seq

In [50]:
# %%time

# X_train, X_val, X_val_seq = split_data(df, validation_cut=2, test_users=test_users)

In [51]:
# %%time

# trs = 20
# counts_aid_filtered = counts_aid.filter(pl.col("count") >= trs).to_pandas()
# pad = len(counts_aid_filtered)

# ordinal_encoder = OrdinalEncoder(handle_unknown="use_encoded_value", unknown_value=pad)
# ordinal_encoder.fit(counts_aid_filtered[config.col_aid].values.reshape(-1, 1))

In [52]:
# %%time

# X_train[config.col_aid] = ordinal_encoder.transform(
#     X_train[config.col_aid].values.reshape(-1, 1)
# ).astype(int)
# X_val_seq[config.col_aid] = ordinal_encoder.transform(
#     X_val_seq[config.col_aid].values.reshape(-1, 1)
# ).astype(int)

In [53]:
# print(X_train.shape)
# X_train = X_train[X_train[config.col_aid] != pad]
# print(X_train.shape)

In [54]:
# %%time

# X_train = pl.from_pandas(X_train)
# X_train = X_train.sort(config.col_ts)

# X_val = pl.from_pandas(X_val)
# X_val = X_val.sort(config.col_ts)

# X_val_seq = pl.from_pandas(X_val_seq)
# X_val_seq = X_val_seq.sort(config.col_ts)

In [55]:
# X_train.filter(pl.col(config.col_user) == 0)

In [56]:
# %%time

# X_train = (
#     X_train.groupby([config.col_user])
#     .agg(
#         [
#             pl.col(config.col_aid).list(),
#         ]
#     ).filter(pl.col(config.col_aid).apply(lambda x: len(x)) >= 2)
# )

# X_val = (
#     X_val.groupby([config.col_user])
#     .agg(
#         [
#             pl.col(config.col_aid).list(),
#         ]
#     )
# )

# X_val_seq = (
#     X_val_seq.groupby([config.col_user])
#     .agg(
#         [
#             pl.col(config.col_aid).list(),
#         ]
#     )
# )

In [57]:
X_train.shape

(67627159, 4)

In [58]:
X_train.filter(pl.col(config.col_user) == 0)

user,user_week,train_seq,target
i64,str,list[i64],i64
0,"""0_31""","[291252, 3657]",102392
0,"""0_31""","[291252, 3657]",367313
0,"""0_31""","[291252, 3657]",175161
0,"""0_31""","[291252, 3657, 367313]",102392
0,"""0_31""","[291252, 3657, 367313]",175161
0,"""0_31""","[291252, 3657, ... 102392]",102392
0,"""0_31""","[291252, 3657, ... 102392]",175161
0,"""0_31""","[392134, 264776, ... 175161]",175161
0,"""0_34""","[148154, 401366, ... 393372]",70209


In [59]:
# dict_map_targets = dict()
# dict_map_targets["clicks"] = 0
# dict_map_targets["carts"] = 1
# dict_map_targets["orders"] = 2

In [60]:
counts_aid["aid"].min(), counts_aid["aid"].max()

(0, 1855602)

In [61]:
pad

413313

In [62]:
# %%time

# df_train = df_train.to_pandas()
# df_train['aid_filtered'] = ordinal_encoder.transform(df_train['aid'].values.reshape(-1, 1)).astype(int)
# df_train = pl.from_pandas(df_train)

In [63]:
# counts_aid_type.filter((pl.col("type") == "orders") & (pl.col("count") >= 1)).select(
#     ["aid"]
# ).unique().shape

In [64]:
# counts_aid_type

In [65]:
counts_aid.tail()

aid,count
i64,u32
96388,1
1112196,1
1570948,1
1153635,1
974369,1


In [66]:
counts_aid.filter(pl.col("aid") == 1)

aid,count
i64,u32
1,19


In [67]:
# len(counts_aid), len(counts_aid_filtered)

In [68]:
# %%time

# token_to_idx = dict()
# # token_to_idx[-1] = 0 # pad
# token_to_idx[0] = 0 # unknown / pad

# idx = 1
# for i in tqdm(range(len(counts_aid["aid"]))):
#     aid = counts_aid["aid"][i]
#     cnt = counts_aid["count"][i]
#     if cnt > 4:
#         token_to_idx[aid] = idx
#         idx += 1
#     # else:
#     #     token_to_idx[aid] = 0

# idx_to_token = {idx: token for token, idx in token_to_idx.items()}

In [69]:
# len(token_to_idx)

In [70]:
# min(list(token_to_idx.values()))

In [71]:
# token_to_idx[0]

In [72]:
# max_len = 10
# # pad = counts_aid["aid"].max() + 1

# # def get_id_of_token(token):
# #     if token in set(list(token_to_idx.keys())[1:]):
# #         idx = token_to_idx[token]
# #     else:
# #         idx = 0  # unknown
# #     return idx


# # def get_token_if_id(idx):
# #     if idx in set(list(idx_to_token.keys())[1:]):
# #         token = idx_to_token[idx]
# #     else:
# #         token = 0  # unknown
# #     return token


# def to_matrix(names, max_len=max_len, pad=pad, dtype="int32", batch_first=True):
#     """Casts a list of names into rnn-digestible matrix"""

#     max_len = max_len or max(map(len, names))
#     names_ix = np.zeros([len(names), max_len], dtype) + pad

#     for i in tqdm(range(len(names))):
#         cur_name = names[i]
#         line_ix = cur_name[-max_len:]  # last max_len elements
#         # cur_name = cur_name[-max_len:]  # last max_len elements
#         # line_ix = [
#         #     get_id_of_token(cur_name[c]) for c in range(len(cur_name)) if c < max_len
#         # ]
#         names_ix[i, -len(line_ix) :] = line_ix

#     if not batch_first:  # convert [batch, time] into [time, batch]
#         names_ix = np.transpose(names_ix)

#     return names_ix

In [73]:
l = [1, 2, 3]
print(utils.split_list_randomly(l=l, seed=1))
print(utils.split_list_randomly(l=l, seed=2))

([1, 2], [3])
([1], [2, 3])


In [74]:
X_train.head()

user,user_week,train_seq,target
i64,str,list[i64],i64
411150,"""411150_31""","[65500, 108855, 45847]",221344
411150,"""411150_31""","[65500, 108855, 45847]",115009
411150,"""411150_31""","[65500, 108855, 45847]",207548
411150,"""411150_31""","[65500, 108855, 45847]",194309
411150,"""411150_31""","[65500, 108855, 45847]",143399


In [75]:
X_train.shape

(67627159, 4)

In [76]:
X_val.head()

user,user_week,train_seq,target
i64,str,list[i64],list[i64]
0,"""0_34""",[397718],[1521766]
3,"""3_31""","[215323, 226403, ... 345815]",[128326]
14,"""14_31""","[96583, 277598, ... 348555]","[284816, 125127]"
15,"""15_31""","[140781, 78558]",[1409430]
19,"""19_31""","[272778, 104886, ... 85086]",[383546]


In [77]:
# %%time

# X_train = X_train.to_pandas()
# X_train[config.col_train_seq] = X_train[config.col_aid].parallel_apply(lambda x: x[:-1])
# X_train[config.col_target] = X_train[config.col_aid].parallel_apply(lambda x: x[-1])
# X_train = pl.from_pandas(X_train)

In [78]:
X_train.head()

user,user_week,train_seq,target
i64,str,list[i64],i64
411150,"""411150_31""","[65500, 108855, 45847]",221344
411150,"""411150_31""","[65500, 108855, 45847]",115009
411150,"""411150_31""","[65500, 108855, 45847]",207548
411150,"""411150_31""","[65500, 108855, 45847]",194309
411150,"""411150_31""","[65500, 108855, 45847]",143399


In [79]:
# %%time

# X_train = X_train.with_column((pl.col(config.col_aid).apply(lambda x: x[:-1]).alias(config.col_train_seq)))
# X_train = X_train.with_column((pl.col(config.col_aid).apply(lambda x: x[-1]).alias(config.col_target)))

In [80]:
# %%time

# X_train['val'] = X_train["val"].progress_apply(lambda x: x[0]) #orig_vals

# col_aids = 'train'
# X_train = pl.from_pandas(X_train)
# X_train = (
#     X_train.groupby(["user", 'val'])
#     .agg(
#         [
#             pl.col(col_aids).list(),
#         ]
#     )
#     # .filter(pl.col(col_aids).apply(lambda x: len(x)) >= 2)
# ).sort('user').to_pandas()

# X_train = X_train[['user', 'train', 'val']]

In [81]:
# X_train.shape

In [82]:
X_train

user,user_week,train_seq,target
i64,str,list[i64],i64
411150,"""411150_31""","[65500, 108855, 45847]",221344
411150,"""411150_31""","[65500, 108855, 45847]",115009
411150,"""411150_31""","[65500, 108855, 45847]",207548
411150,"""411150_31""","[65500, 108855, 45847]",194309
411150,"""411150_31""","[65500, 108855, 45847]",143399
411150,"""411150_31""","[65500, 108855, 45847]",105033
411150,"""411150_31""","[65500, 108855, 45847]",347498
411150,"""411150_31""","[65500, 108855, 45847]",412910
411150,"""411150_31""","[65500, 108855, 45847]",397135


In [83]:
# X_train.explode(config.col_target)[config.col_target].value_counts()

In [84]:
X_train[config.col_target].value_counts()

target,counts
i64,u32
40320,91
92352,84
150208,46
3264,12
378944,38
156736,37
119552,74
60160,89
63296,12


In [85]:
# %%time

# X_train = X_train.to_pandas()

# list_train = list()
# list_target = list()
# list_user = list()

# n_iter = 15
# for i in tqdm(range(len(X_train))):
#     row = X_train.iloc[i]
#     row_train = list(row[config.col_train_seq])
#     target_cur_main = row[config.col_target]
#     len_row_train = len(row_train)
#     if len_row_train == 1 and target_cur_main < pad:
#         list_train.append(tuple(row_train))
#         list_target.append(target_cur_main)
#         list_user.append(row[config.col_user])
#     else:
#         n_iter_cur = min(n_iter, len_row_train - 1)
#         # train_cur_main = utils.truncate_list(l=row_train, max_len=config.max_len, from_start=False)
#         if target_cur_main < pad:
#             list_train.append(tuple(row_train))
#             list_target.append(target_cur_main)
#             list_user.append(row[config.col_user])

#         for seed in range(n_iter_cur):
#             train_cur, target_cur = utils.split_list_randomly(l=row_train, seed=seed)
#             target_cur = target_cur[0]
#             if target_cur < pad:
#                 # train_cur = utils.truncate_list(l=train_cur, max_len=config.max_len, from_start=False)
#                 list_train.append(tuple(train_cur))
#                 list_target.append(target_cur)
#                 list_user.append(row[config.col_user])

# X_train = pd.DataFrame()
# X_train[config.col_user] = list_user
# X_train[config.col_train_seq] = list_train
# X_train[config.col_target] = list_target
# X_train.drop_duplicates(inplace=True)
# X_train.reset_index(drop=True, inplace=True)

# X_train = pl.from_pandas(X_train)

In [86]:
# X_train.head()

In [87]:
# X_train.shape

In [88]:
# %%time

# X_train = X_train.to_pandas()
# X_train[config.col_train_seq] = X_train[config.col_train_seq].progress_apply(lambda x: utils.truncate_list(l=list(x), max_len=config.max_len, from_start=False))
# X_train = pl.from_pandas(X_train)

In [89]:
# %%time

# X_train = X_train.to_pandas()
# X_train[config.col_target] = X_train[config.col_target].parallel_apply(lambda x: utils.truncate_list(l=list(x), max_len=20, from_start=True))
# X_train = pl.from_pandas(X_train)

In [90]:
X_train.head()

user,user_week,train_seq,target
i64,str,list[i64],i64
411150,"""411150_31""","[65500, 108855, 45847]",221344
411150,"""411150_31""","[65500, 108855, 45847]",115009
411150,"""411150_31""","[65500, 108855, 45847]",207548
411150,"""411150_31""","[65500, 108855, 45847]",194309
411150,"""411150_31""","[65500, 108855, 45847]",143399


In [91]:
# %%time

# X_val = X_val.to_pandas()
# X_val[config.col_train_seq] = X_val[config.col_train_seq].parallel_apply(
#     lambda x: utils.truncate_list(l=list(x), max_len=config.max_len, from_start=False)
# )
# X_val = pl.from_pandas(X_val)

In [92]:
# %%time

# X_val = X_val.to_pandas()
# X_val[config.col_target] = X_val[config.col_target].parallel_apply(
#     lambda x: utils.truncate_list(l=list(x), max_len=20, from_start=True)
# )
# X_val = pl.from_pandas(X_val)

In [93]:
X_val.head()

user,user_week,train_seq,target
i64,str,list[i64],list[i64]
0,"""0_34""",[397718],[1521766]
3,"""3_31""","[215323, 226403, ... 345815]",[128326]
14,"""14_31""","[96583, 277598, ... 348555]","[284816, 125127]"
15,"""15_31""","[140781, 78558]",[1409430]
19,"""19_31""","[272778, 104886, ... 85086]",[383546]


In [94]:
# X_val = X_val.join(
#     X_val_seq[[config.col_user, config.col_train_seq]], how="inner", on=config.col_user
# )

In [95]:
# X_val = X_val.rename({config.col_aid: config.col_target})

In [96]:
# X_val.filter(pl.col(config.col_user) == 13833088)

In [97]:
# %%time

# X_val["train"] = X_val["train"].progress_apply(lambda x: utils.truncate_list(l=list(x), max_len=max_len, from_start=False))
# X_val["val"] = X_val["val"].progress_apply(lambda x: x[0]) #orig_vals

In [98]:
# %%time

# X_val = X_val.explode('train')
# X_val['train'] = ordinal_encoder.transform(X_val['train'].values.reshape(-1, 1)).astype(int)

# col_aids = 'train'
# X_val = pl.from_pandas(X_val)
# X_val = (
#     X_val.groupby(["user", 'val'])
#     .agg(
#         [
#             pl.col(col_aids).list(),
#         ]
#     )
#     # .filter(pl.col(col_aids).apply(lambda x: len(x)) >= 2)
# ).sort('user').to_pandas()

# X_val = X_val[['user', 'train', 'val']]

In [99]:
# %%time

# df_train_clicks["val"] = df_train_clicks["val"].progress_apply(lambda x: x[0])

In [100]:
# df_train_clicks["val"].value_counts()

In [101]:
# df_train_clicks.head()

In [102]:
# df_train[col_aids].value_counts().sort("counts", reverse=True)

In [103]:
# df_train_clicks.drop([col_aids, "split"], axis=1, inplace=True)

In [104]:
# %%time

# df_train_clicks = df_train_clicks.with_columns(
#     [pl.col("aid").apply(lambda x: utils.split_list_randomly(list(x))).alias("split")]
# )

In [105]:
# %%time

# df_train_clicks = df_train_clicks.with_columns(
#     [
#         pl.col("split").apply(lambda x: x[0]).alias("train"),
#         pl.col("split").apply(lambda x: x[1]).alias("val"),
#     ]
# )

# for col in ["aid", "split"]:
#     df_train_clicks.drop_in_place(col)

# df_train_clicks = df_train_clicks.sort("user")

In [106]:
# %%time

# df_train_clicks_mapped = df_train_clicks.clone()
# df_train_clicks_mapped = df_train_clicks_mapped.to_pandas()

In [107]:
# %%time

# df_train_clicks_mapped["val"] = df_train_clicks_mapped["val"].parallel_apply(
#     lambda x: get_id_of_token(x[0])
# )

In [108]:
# shape_orig = df_train_clicks_mapped.shape
# df_train_clicks_mapped = df_train_clicks_mapped[
#     df_train_clicks_mapped["val"] != 0
# ].reset_index(drop=True)
# print(shape_orig, "->", df_train_clicks_mapped.shape)

In [109]:
# (12773, 4)

In [110]:
# %%time

# df_train_clicks["train"] = df_train_clicks["train"].progress_apply(lambda x: utils.truncate_list(list(x), max_len=max_len))

In [111]:
# %%time

# df_train_clicks["train"] = list(to_matrix(df_train_clicks["train"]))

In [112]:
# df_train_clicks["val"].value_counts()

In [113]:
# df_train_clicks.head()

In [114]:
# df_train_clicks["train"][0]

In [115]:
# %%time

# df_train_clicks["train"] = df_train_clicks["train"].parallel_apply(
#     lambda x: to_matrix([x])[0]
# )

In [116]:
# %%time

# df_train_clicks_mapped = df_train_clicks.clone()
# df_train_clicks_mapped = df_train_clicks_mapped.with_columns(
#     [
#         pl.col("train").apply(lambda x: to_matrix(list(x))[0]).alias("train"),
#         pl.col("val").apply(lambda x: to_matrix(list('val')[:1])[0]).alias("val"),
#     ]
# )
# df_train_clicks_mapped = df_train_clicks_mapped.sort("user")

In [117]:
# class Dataset_RNN(Dataset):
#     def __init__(self, df, mode):

#         self.df = df  # .reset_index(drop=True)
#         self.mode = mode

#     def __len__(self):
#         return len(self.df)

#     def __getitem__(self, index):
#         row = self.df.iloc[index]
#         user = row["user"]
#         aids_train = row["train"]
#         # aids_train = to_matrix([aids_train])[0]
#         aids_val = row["val"]
#         # aids_val = to_matrix([aids_val])[0]

#         if self.mode == "test":
#             return torch.tensor(aids_train).long()
#         else:
#             return torch.tensor(aids_train).long(), torch.tensor(aids_val).float()

In [118]:
class Attention(nn.Module):
    def __init__(self, hidden_size, batch_first=False):
        super(Attention, self).__init__()

        self.hidden_size = hidden_size
        self.batch_first = batch_first

        self.att_weights = nn.Parameter(
            torch.Tensor(1, hidden_size), requires_grad=True
        )

        stdv = 1.0 / np.sqrt(self.hidden_size)
        for weight in self.att_weights:
            nn.init.uniform_(weight, -stdv, stdv)

    def get_mask(self):
        pass

    def forward(self, inputs, lengths):
        if self.batch_first:
            batch_size, max_len = inputs.size()[:2]
        else:
            max_len, batch_size = inputs.size()[:2]

        # print(f"inputs: {inputs.size()}")

        # apply attention layer
        weights = torch.bmm(
            inputs,
            self.att_weights.permute(1, 0)  # (1, hidden_size) --> (hidden_size, 1)
            .unsqueeze(0)  # (1, hidden_size, 1)
            .repeat(batch_size, 1, 1),  # (batch_size, hidden_size, 1)
        )

        # print(f"weights: {weights.size()}")

        attentions = torch.softmax(F.relu(weights.squeeze()), dim=-1)
        # print(f"attentions: {attentions.size()}")

        # create mask based on the sentence lengths
        mask = torch.ones(attentions.size(), requires_grad=True).to(device)
        # print(f"mask: {mask.size()}")
        if batch_size == 1:
            mask = mask.view(1, -1)
        for i, l in enumerate(lengths):  # skip the first sentence
            if l < max_len:
                mask[i, l:] = 0

        # apply mask and renormalize attention scores (weights)
        masked = attentions * mask
        _sums = masked.sum(-1).unsqueeze(-1)  # sums per row

        attentions = masked.div(_sums)

        # apply attention weights
        weighted = torch.mul(inputs, attentions.unsqueeze(-1).expand_as(inputs))
        # print(f"weighted: {weighted.size()}")

        # get the final fixed vector representations of the sentences
        representations = weighted.sum(1).squeeze()
        # print(f"representations: {representations.size()}")

        return representations, attentions

In [119]:
# min(600, round(1.6 * n_cat**0.56))

n_vocab = pad + 1
# hidden_dim = 64
# embedding_dim = min(600, round(1.6 * pad**0.56))
# embedding_dim = 64

In [120]:
class RRNN(nn.Module):
    def __init__(
        self,
        n_vocab=n_vocab,
        hidden_dim=config.hidden_dim,
        embedding_dim=config.embedding_dim,
        dropout=0.1,
    ):
        super(RRNN, self).__init__()
        self.embedding_dim = embedding_dim
        self.embeddings = nn.Embedding(n_vocab, embedding_dim, sparse=False)
        self.hidden_dim = hidden_dim
        self.out_features_num = n_vocab
        self.dropout0 = nn.Dropout(dropout, inplace=True)
        self.lstm_desc = nn.GRU(
            embedding_dim,
            hidden_dim,
            dropout=dropout,
            num_layers=1,
            bidirectional=False,
            batch_first=True,
        )

        self.pooling_result_dimension = self.hidden_dim * 1
        self.dim_after_concat = self.pooling_result_dimension * 1  # + 6 + 14 + 19

        self.attention_descs = Attention(self.hidden_dim * 1, batch_first=True)
        self.bn1 = nn.BatchNorm1d(self.dim_after_concat)
        self.dropout1 = nn.Dropout(dropout, inplace=True)
        self.fc1 = nn.Linear(self.dim_after_concat, self.embedding_dim)
        self.dropout2 = nn.Dropout(dropout, inplace=True)
        self.relu1 = nn.ReLU()
        self.fc2 = nn.Linear(self.embedding_dim, self.out_features_num)

        self.fc2.weight = self.embeddings.weight

        # self.category_embedding = self._create_embedding_projection(cardinality=10, embed_size=6, padding_idx=None, scale_grad_by_freq=False)
        # self.subcategory_embedding = self._create_embedding_projection(cardinality=50, embed_size=14, padding_idx=None, scale_grad_by_freq=False)
        # self.region_embedding = self._create_embedding_projection(cardinality=85, embed_size=19, padding_idx=None, scale_grad_by_freq=False)

    def _create_embedding_projection(
        self,
        cardinality,
        embed_size,
        add_missing=True,
        padding_idx=0,
        scale_grad_by_freq=False,
    ):
        add_missing = add_missing * 1
        return nn.Embedding(
            num_embeddings=cardinality + add_missing,
            embedding_dim=embed_size,
            padding_idx=padding_idx,
            scale_grad_by_freq=scale_grad_by_freq,
        )

    def forward(self, aids, lens):
        batch_size = aids.shape[0]

        embedded_aids = self.embeddings(aids)
        embedded_aids = self.dropout0(embedded_aids)
        pad_embed_pack = torch.nn.utils.rnn.pack_padded_sequence(
            embedded_aids, list(lens), batch_first=True, enforce_sorted=False
        )
        states_aids, hh_aids = self.lstm_desc(pad_embed_pack)

        states_aids, lens_unpacked = torch.nn.utils.rnn.pad_packed_sequence(
            states_aids, batch_first=True
        )

        lengths_aids = [x_.size(1) for x_ in states_aids]
        atten_aids, _ = self.attention_descs(states_aids, lengths_aids)
        atten_aids = atten_aids.view(batch_size, -1)

        features = torch.cat(
            [
                # subcategory_embed, category_embed, region_embed,
                atten_aids
            ],
            dim=-1,
        )
        features = self.bn1(features)
        features = self.dropout1(features)
        features = features.view(features.size(0), -1)
        features = self.fc1(features)
        features = self.dropout2(features)
        features = self.relu1(features)
        features = self.fc2(features)
        # features = F.normalize(features)

        return features

In [121]:
%%time
X_train = X_train.to_pandas()
X_val = X_val.to_pandas()

X_train_to_dataset = pd.DataFrame(X_train[config.col_train_seq].tolist(), index=X_train.index)
X_train_to_dataset.columns = ['pos' + '_' + str(pos) for pos in X_train_to_dataset.columns]
X_train_to_dataset[config.col_target] = X_train[config.col_target]
X_train_to_dataset[config.col_user] = X_train[config.col_user]
X_train_to_dataset[config.col_n_week] = X_train[config.col_user + '_week'].progress_apply(lambda x: int(x.split('_')[-1]))
X_train_to_dataset.to_parquet('train.parquet', index=False)

cols_target_train = [col for col in X_train_to_dataset.columns if "target" in col]
cols_pos = [col for col in X_train_to_dataset.columns if "pos" in col]

del X_train, X_train_to_dataset
gc.collect()
# X_train_target_to_dataset = pd.DataFrame(X_train_to_dataset[config.col_target].tolist(), index=X_train_to_dataset.index)
# X_train_target_to_dataset.columns = ['target' + '_' + str(pos) for pos in X_train_target_to_dataset.columns]
# X_train_to_dataset[X_train_target_to_dataset.columns] = X_train_target_to_dataset[X_train_target_to_dataset.columns]
# X_train_to_dataset.drop([config.col_target], axis=1, inplace=True)

##############

X_val_to_dataset = pd.DataFrame(X_val[config.col_train_seq].tolist(), index=X_val.index)
X_val_to_dataset.columns = ['pos' + '_' + str(pos) for pos in X_val_to_dataset.columns]
X_val_to_dataset[config.col_target] = X_val[config.col_target]
X_val_to_dataset[config.col_user] = X_val[config.col_user]
X_val_to_dataset[config.col_n_week] = X_val[config.col_user + '_week'].progress_apply(lambda x: int(x.split('_')[-1]))

del X_val
gc.collect()

X_val_target_to_dataset = pd.DataFrame(X_val_to_dataset[config.col_target].tolist(), index=X_val_to_dataset.index)
X_val_target_to_dataset.columns = ['target' + '_' + str(pos) for pos in X_val_target_to_dataset.columns]
X_val_to_dataset[X_val_target_to_dataset.columns] = X_val_target_to_dataset[X_val_target_to_dataset.columns]
X_val_to_dataset.drop([config.col_target], axis=1, inplace=True)
X_val_to_dataset.to_parquet('valid.parquet', index=False)
# X_train = pl.from_pandas(X_train)
# X_val = pl.from_pandas(X_val)

cols_target_val = [col for col in X_val_to_dataset.columns if "target" in col]

del X_val_to_dataset, X_val_target_to_dataset
gc.collect()

  0%|          | 0/67627159 [00:00<?, ?it/s]

  0%|          | 0/694051 [00:00<?, ?it/s]

CPU times: user 5min 26s, sys: 34.8 s, total: 6min 1s
Wall time: 5min 58s


0

In [122]:
cols_target_val

['target_0',
 'target_1',
 'target_2',
 'target_3',
 'target_4',
 'target_5',
 'target_6',
 'target_7',
 'target_8',
 'target_9',
 'target_10',
 'target_11',
 'target_12',
 'target_13',
 'target_14',
 'target_15',
 'target_16',
 'target_17',
 'target_18',
 'target_19']

In [123]:
cols_target_train

['target']

In [124]:
# X_train_to_dataset.head()

In [125]:
# X_train_to_dataset[["user", "n_week"]]

In [126]:
# X_train_to_dataset.shape

In [127]:
# X_train_to_dataset[X_train_to_dataset["pos_0"].isnull()]

In [128]:
# X_val_to_dataset.head()

In [129]:
# X_val_to_dataset[X_val_to_dataset["pos_0"].isnull()]

In [130]:
# X_val_to_dataset[["user", "n_week"]]

In [131]:
# %%time

# X_train_to_dataset.to_parquet('train.parquet', index=False)
# X_val_to_dataset.to_parquet('valid.parquet', index=False)

In [132]:
%%time

train_ds = Dataset_merlin('train.parquet')
train_dl_merlin = Loader(dataset=train_ds, batch_size=config.batch_size_train, shuffle=True)

valid_ds = Dataset_merlin('valid.parquet')
valid_dl_merlin = Loader(dataset=valid_ds, batch_size=config.batch_size_infer, shuffle=False)

CPU times: user 9.44 s, sys: 15 s, total: 24.4 s
Wall time: 3.42 s


In [133]:
def get_matrix_and_target_from_batch(
    batch,
    cols_pos=cols_pos,
    # cols_target_train=cols_target_train,
    cols_target_val=cols_target_val,
    pad=pad,
    mod="inference",
):
    assert mod in ["inference", "train", "valid"]

    batch_matrix = torch.zeros([len(batch[cols_pos[0]]), len(cols_pos)])
    for i in range(len(cols_pos)):
        batch_matrix[:, i] = torch.reshape(batch[cols_pos[i]], (1, -1))
    batch_matrix = [
        torch.tensor([int(val) for val in batch_matrix[i] if not math.isnan(val)])
        for i in range(len(batch_matrix))
    ]
    lens = torch.tensor(list(map(len, batch_matrix)))

    batch_matrix = torch.nn.utils.rnn.pad_sequence(
        batch_matrix, batch_first=True, padding_value=pad  # + 1
    )

    users = batch[config.col_user].squeeze(1).tolist()
    n_weeks = batch[config.col_n_week].squeeze(1).tolist()

    if mod == "train":
        target = batch[config.col_target]
    elif mod == "valid":
        cols_target = cols_target_val
        target = torch.zeros([len(batch[cols_target[0]]), len(cols_target)])
        for i in range(len(cols_target)):
            target[:, i] = torch.reshape(batch[cols_target[i]], (1, -1))
        target = [
            torch.tensor([int(val) for val in target[i] if not math.isnan(val)])
            for i in range(len(target))
        ]
    elif mod == "inference":
        target = None
    return batch_matrix, target, lens, users, n_weeks

In [134]:
%%time

for batch, _ in tqdm(train_dl_merlin):
    # batch_matrix, batch["val"], lens = get_matrix_and_target_from_batch(batch, cols_pos=cols_pos, pad=pad)
    continue
    # break
# 5 min

  0%|          | 0/264169 [00:00<?, ?it/s]

CPU times: user 40.3 s, sys: 31 s, total: 1min 11s
Wall time: 46.5 s


In [135]:
for batch, _ in tqdm(train_dl_merlin):
    break

  0%|          | 0/264169 [00:00<?, ?it/s]

In [136]:
# %%time

# batch_matrix = np.zeros([len(batch[cols_pos[0]]), len(cols_pos)])
# for i in range(len(cols_pos)):
#     batch_matrix[:, i] = torch.reshape(batch[cols_pos[i]], (1, -1))
# batch_matrix = [
#         torch.tensor([int(val) for val in batch_matrix[i] if not np.isnan(val)])
#         for i in range(len(batch_matrix))
#     ]

In [137]:
# def get_matrix_and_target_from_batch(batch, cols_pos=cols_pos):
#     batch_matrix = torch.zeros([len(batch[cols_pos[0]]), len(cols_pos)])
#     for i in range(len(cols_pos)):
#         batch_matrix[:, i] = torch.reshape(batch[cols_pos[i]], (1, -1))
#     # batch_target = torch.reshape(batch["val"], (1, -1))
#     return batch_matrix, batch["val"]

In [138]:
DEBUG = False


def train_func(
    train_loader, model, criterion, opts: list, amp=False, num_batches_to_check=1000
):
    model.train()
    bar = tqdm(train_loader)
    losses = list()
    if amp:
        scaler = torch.cuda.amp.GradScaler()

    start_time = time.perf_counter()
    batches_passed = 0
    for batch_idx, (batch, _) in enumerate(bar):
        if num_batches_to_check:
            if batch_idx % num_batches_to_check == 0 and batch_idx > 0:
                batches_passed += num_batches_to_check
                end_time = time.perf_counter()
                total_time = end_time - start_time
                print(f"batches passed: {batches_passed}")
                print(
                    f"last {num_batches_to_check} batches took {total_time / 60 :.4f} minutes"
                )
                print()
                start_time = time.perf_counter()

        aids, targets, lens, users, n_weeks = get_matrix_and_target_from_batch(
            batch, mod="train"
        )
        aids = aids.to(device).long()
        lens = lens.to(device).long()
        # category = category.to(device).long()
        # subcategory = subcategory.to(device).long()
        # region = region.to(device).long()
        # target_matrix = torch.zeros(len(targets), pad + 1)
        # for i in range(len(targets)):
        #     target_matrix[i] = torch.zeros(
        #         targets[i].unsqueeze(0).size(0), pad + 1
        #     ).scatter_(1, targets[i].unsqueeze(0), 1.0)
        # target_matrix = target_matrix.to(device).long()
        # target_matrix = target_matrix.squeeze(1)

        targets = targets.to(device).long()
        targets = targets.squeeze(1)

        if DEBUG and batch_idx == 2:
            print("Debug Mode. Only train on first 1 batches.")
            break

        if amp:
            with torch.cuda.amp.autocast():
                logits = model(aids, lens)
                loss = criterion(logits, targets)
                scaler.scale(loss).backward()  # loss.backward()
                for optimizer in opts:
                    scaler.step(optimizer)  # optimizer.step()
                    scaler.update()
        else:
            # print(aids)
            # print(lens)
            logits = model(aids, lens)
            # print(logits.size())
            loss = criterion(logits, targets)
            loss.backward()
        for optimizer in opts:
            optimizer.step()

        for optimizer in opts:
            optimizer.zero_grad()

        losses.append(loss.item())
        # smooth_loss = np.mean(losses[-30:])

        # bar.set_description(f"smooth_loss: {smooth_loss:.5f}")

    loss_train = np.mean(losses)
    print("loss_train:", loss_train)
    return loss_train


def do_inverse_transform(array, ordinal_encoder=ordinal_encoder):
    result = ordinal_encoder.inverse_transform(array.reshape(-1, 1)).reshape(1, -1)[0]
    result = [val for val in result if val is not None]
    return result


def flatten_list(l):
    return [item for sublist in l for item in sublist]


def valid_func(valid_loader, model, criterion):
    model.eval()

    bar = tqdm(valid_loader)
    list_preds = list()
    list_targets = list()
    list_users = list()
    list_n_weeks = list()

    losses = list()
    with torch.no_grad():
        for batch_idx, (batch, _) in enumerate(bar):
            aids, targets, lens, users, n_weeks = get_matrix_and_target_from_batch(
                batch, mod="valid"
            )
            aids = aids.to(device).long()
            lens = lens.to(device).long()
            # category = category.to(device).long()
            # subcategory = subcategory.to(device).long()
            # region = region.to(device).long()
            # targets = targets.to(device).long()
            # targets = targets.squeeze(1)

            logits = model(aids, lens)
            # loss = criterion(logits, targets)
            # losses.append(loss.item())

            # PREDS += [torch.argmax(logits, 1).detach().cpu()]
            k = 20
            preds_cur = torch.topk(logits, k).indices.detach().cpu().numpy()
            # print(preds_cur.shape)
            # print(preds_cur)
            # preds_cur = [np.argpartition(-pred, k)[:k] for pred in preds_cur]
            # preds_cur = [
            #     do_inverse_transform(pred, ordinal_encoder=ordinal_encoder)
            #     for pred in preds_cur
            # ]
            list_preds.append(preds_cur)
            targets = [target.tolist() for target in targets]
            list_targets.append(targets)
            list_users.append(users)
            list_n_weeks.append(n_weeks)

            # bar.set_description(f"loss: {loss.item():.5f}")

    # loss_valid = np.mean(losses)
    # print("loss_valid:", loss_valid)

    # PREDS = torch.cat(PREDS).cpu().tolist()
    list_users = flatten_list(list_users)
    list_preds = flatten_list(list_preds)
    list_targets = flatten_list(list_targets)
    list_n_weeks = flatten_list(list_n_weeks)

    # TARGETS = torch.cat(TARGETS).cpu().tolist()
    # print(len(PREDS), len(TARGETS))
    # print(TARGETS)
    # print(PREDS)
    df_val_targets = pd.DataFrame(
        {"session": list_users, "n_weeks": list_n_weeks, "targets": list_targets}
    )

    df_val = pd.DataFrame(
        {"session": list_users, "n_weeks": list_n_weeks, "preds": list_preds}
    )
    df_val = df_val.explode("preds")

    df_val["preds"] = ordinal_encoder.inverse_transform(
        df_val["preds"].values.reshape(-1, 1)
    ).reshape(1, -1)[0]
    df_val = df_val.dropna()
    df_val["preds"] = df_val["preds"].astype(int)
    df_val = pl.from_pandas(df_val)
    df_val = (
        (
            df_val.groupby(["session", "n_weeks"]).agg(
                [
                    pl.col("preds").list(),
                ]
            )
        )
        .sort("session")
        .to_pandas()
    )
    df_val = df_val_targets.merge(df_val, how="inner", on=["session", "n_weeks"])

    list_preds = df_val["preds"].tolist()
    list_preds = [list(pred) for pred in list_preds]

    list_targets = df_val["targets"].tolist()
    list_targets_top_1 = [target[0] for target in list_targets]

    metric_total = utils.recall_several_targets(list_preds, list_targets)
    print("metric_total:", metric_total)

    metric_top_1 = utils.recall_one_val(list_preds, list_targets_top_1)
    print("metric_top_1:", metric_top_1)

    return df_val


# def inference_func(inference_loader, model):
#     model.eval()

#     bar = tqdm(inference_loader)
#     preds = list()

#     with torch.no_grad():
#         for batch_idx, (descs, category, subcategory, region) in enumerate(bar):
#             descs = descs.to(device).long()
#             category = category.to(device).long()
#             subcategory = subcategory.to(device).long()
#             region = region.to(device).long()

#             logits = model(descs, category, subcategory, region)

#             preds += [logits.detach().cpu()]

#     preds = torch.cat(preds).cpu().numpy()
#     preds = 1.0 / (1.0 + np.exp(-preds))
#     preds = np.concatenate(preds, axis=0)
#     return preds

In [139]:
out_features_num = pad + 1
out_features_num

413314

In [140]:
model = RRNN()
model.to(device)

/usr/bin/nvidia-modprobe: unrecognized option: "-f"

ERROR: Invalid commandline, please run `/usr/bin/nvidia-modprobe --help` for usage information.


/usr/bin/nvidia-modprobe: unrecognized option: "-f"

ERROR: Invalid commandline, please run `/usr/bin/nvidia-modprobe --help` for usage information.




RRNN(
  (embeddings): Embedding(413314, 64)
  (dropout0): Dropout(p=0.1, inplace=True)
  (lstm_desc): GRU(64, 32, batch_first=True, dropout=0.1)
  (attention_descs): Attention()
  (bn1): BatchNorm1d(32, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
  (dropout1): Dropout(p=0.1, inplace=True)
  (fc1): Linear(in_features=32, out_features=64, bias=True)
  (dropout2): Dropout(p=0.1, inplace=True)
  (relu1): ReLU()
  (fc2): Linear(in_features=64, out_features=413314, bias=True)
)

In [141]:
params_sparsed = list()
params_not_sparsed = list()
cnt = 0
for param in model.parameters():
    if cnt == 0:
        params_sparsed.append(param)
    else:
        params_not_sparsed.append(param)
    cnt += 1

In [142]:
params_sparsed[0].shape

torch.Size([413314, 64])

In [143]:
class MultipleOptimizer:
    def __init__(self, *optimizers):
        self.optimizers = optimizers

    def zero_grad(self):
        for optimizer in self.optimizers:
            optimizer.zero_grad()

    def step(self):
        for optimizer in self.optimizers:
            optimizer.step()


class MultipleScheduler:
    def __init__(self, *schedulers):
        self.schedulers = schedulers

    def step(self):
        for scheduler in self.schedulers:
            scheduler.step()

In [144]:
init_lr = 0.0001

criterion = nn.CrossEntropyLoss(ignore_index=pad)
# criterion = nn.MultiLabelMarginLoss()
optimizer = optim.Adam(model.parameters(), lr=init_lr)
# optimizer_sparsed = optim.SparseAdam(params_sparsed, lr=init_lr)
# optimizer_not_sparsed = optim.Adam(params_not_sparsed, lr=init_lr)
opts = [optimizer]  # , optimizer_sparsed, optimizer_not_sparsed
# opt = MultipleOptimizer(optimizer_sparsed, optimizer_not_sparsed)

# scheduler_sparsed = torch.optim.lr_scheduler.CosineAnnealingLR(
#     optimizer_sparsed, n_epochs
# )
# scheduler_not_sparsed = torch.optim.lr_scheduler.CosineAnnealingLR(
#     optimizer_sparsed, n_epochs
# )
# scheduler = MultipleScheduler(scheduler_sparsed, scheduler_not_sparsed)

In [145]:
%%time

for epoch in tqdm(range(config.n_epochs), desc="epochs"):

    print(f"Epoch {epoch + 1} started")
    torch.cuda.empty_cache()

    # print("LR:", scheduler.schedulers[0].state_dict()["_last_lr"][0])
    loss_train = train_func(train_dl_merlin, model, criterion, opts, num_batches_to_check=None)
    # scheduler.step()

    df_val = valid_func(valid_dl_merlin, model, criterion)
    # X_val['preds'] = preds_local
    # aucs = list()
    # for category, cur_df in X_val.groupby('category'):
    #     cur_auc = roc_auc_score(cur_df['is_bad'], cur_df['preds'])
    #     aucs.append(cur_auc)
    # print()
    # print(aucs)
    # print('local mean_auc:', np.mean(aucs))

    # loss_valid, preds = valid_func(valid_loader, model, criterion)
    # df_val['preds'] = preds
    # aucs = list()
    # for category, cur_df in df_val.groupby('category'):
    #     cur_auc = roc_auc_score(cur_df['is_bad'], cur_df['preds'])
    #     aucs.append(cur_auc)
    # print()
    # print(aucs)
    # print('mean_auc:', np.mean(aucs))

    torch.save(
        model.state_dict(),
        f"{config.model_name}_epoch{epoch + 1}.pth",
    )

    print("Checkpoint saved!")
    print()
    print("*" * 90)
    print()
    print()

# del model
torch.cuda.empty_cache()
gc.collect()

epochs:   0%|          | 0/10 [00:00<?, ?it/s]

Epoch 1 started


  0%|          | 0/264169 [00:00<?, ?it/s]

loss_train: 11.451764408406142


  0%|          | 0/2712 [00:00<?, ?it/s]

metric_total: 0.07194903123504205
metric_top_1: 0.08011370922309743
Checkpoint saved!

******************************************************************************************


Epoch 2 started


... (18 outputs left out)

  0%|          | 0/2712 [00:00<?, ?it/s]

metric_total: 0.17599173356865289
metric_top_1: 0.1942436506827308
Checkpoint saved!

******************************************************************************************


Epoch 7 started


  0%|          | 0/264169 [00:00<?, ?it/s]

KeyboardInterrupt: 

In [146]:
val = X_val_seq[[config.col_user, config.col_train_seq]].clone()
val = val.join(
    pl.from_pandas(df_val), left_on=config.col_user, right_on="session"
).to_pandas()
val["len_seq"] = val[config.col_train_seq].apply(len)
val["target_top1"] = [targets[0] for targets in val["targets"]]
val["targets"] = [targets.tolist() for targets in val["targets"]]
val["preds"] = [preds.tolist() for preds in val["preds"]]

NameError: name 'X_val_seq' is not defined

In [ ]:
val.head()

In [ ]:
%%time

list_len_seq = list()
list_recall_several = list()
list_recall_top1 = list()

for len_seq, df_cur in tqdm(val.groupby('len_seq')):
    list_len_seq.append(len_seq)
    recall_several = utils.recall_several_targets(list(df_cur['preds']), list(df_cur['targets']))
    list_recall_several.append(recall_several)
    recall_top1 = utils.recall_one_val(list(df_cur['preds']), list(df_cur['target_top1']))
    list_recall_top1.append(recall_top1)

In [ ]:
list_len_seq

In [ ]:
list_recall_several

In [ ]:
list_recall_top1

In [ ]:
# 0.10479944304610936

In [ ]:
# metric_total: 0.042083333333333334 1 ep
# metric_total: 0.06625 1 ep 5 iter
# metric_total: 0.059583333333333335 1 ep 10 iter
# metric_total: 0.0725 1 ep 15 iter

In [ ]:
X_val[config.col_target].value_counts().head(10)

In [ ]:
X_val = X_val.to_pandas()

In [ ]:
X_val.shape

In [ ]:
X_val.tail()

In [ ]:
stop

In [ ]:
%%time

PATH_DATA = '../data/02_intermediate/'
# df_train = pl.read_parquet(PATH_DATA + 'df_train_n_session.parquet')
df_test = pl.read_parquet(PATH_DATA + 'df_test_n_session.parquet')
# df_train = df_train.filter(pl.col("user") % 100 == 0)
# df_train = df_train.to_pandas()
df_test = df_test.to_pandas()
# df_test = catalog.load("df_test")

# df_train.name = "df_train"
df_test.name = "df_test"

# df_train = utils.make_date_stats(df_train)
df_test = utils.make_date_stats(df_test)

# df_train.drop(['date_day'], axis=1, inplace=True)
df_test.drop(['date_day'], axis=1, inplace=True)

# df_train = pl.from_pandas(df_train)
df_test = pl.from_pandas(df_test)

In [ ]:
df_test = df_test.sort(config.col_ts)

In [ ]:
df_test = df_test.to_pandas()
df_test[config.col_aid] = ordinal_encoder.transform(
    df_test[config.col_aid].values.reshape(-1, 1)
).astype(int)
df_test = pl.from_pandas(df_test)

In [ ]:
%%time

df_test = (
    df_test.groupby([config.col_user])
    .agg(
        [
            pl.col(config.col_aid).list().alias(config.col_train_seq),
        ]
    )
)

In [ ]:
%%time

df_test = df_test.to_pandas()

to_dataset = pd.DataFrame(df_test[config.col_train_seq].tolist(), index=df_test.index)
to_dataset.columns = ['pos' + '_' + str(pos) for pos in to_dataset.columns]
to_dataset[config.col_user] = df_test[config.col_user]

df_test = pl.from_pandas(df_test)

cols_pos = [col for col in to_dataset.columns if "pos" in col]

PATH_DATASET = 'to_dataset.parquet'

to_dataset.to_parquet(PATH_DATASET, index=False)

dataset = Dataset_merlin(PATH_DATASET)
dl_merlin = Loader(dataset=dataset, batch_size=config.batch_size_infer, shuffle=False)

In [ ]:
def inference_func(loader, model, num_candidates=20):
    model.eval()

    bar = tqdm(loader)
    list_preds = list()
    list_users = list()

    with torch.no_grad():
        for batch_idx, (batch, _) in enumerate(bar):
            aids, _, lens, users = get_matrix_and_target_from_batch(
                batch, mod="inference"
            )
            aids = aids.to(device).long()
            lens = lens.to(device).long()
            logits = model(aids, lens)
            preds_cur = (
                torch.topk(logits, num_candidates).indices.detach().cpu().numpy()
            )
            list_preds.append(preds_cur)
            list_users.append(users)

    list_preds = utils.flatten_list(list_preds)
    list_users = utils.flatten_list(list_users)

    df_preds = pd.DataFrame({"session": list_users, "preds": list_preds})
    df_preds = df_preds.explode("preds")

    df_preds["preds"] = ordinal_encoder.inverse_transform(
        df_preds["preds"].values.reshape(-1, 1)
    ).reshape(1, -1)[0]
    df_preds = df_preds.dropna()
    df_preds["preds"] = df_preds["preds"].astype(int)
    df_preds = pl.from_pandas(df_preds)
    df_preds = (
        (
            df_preds.groupby(["session"]).agg(
                [
                    pl.col("preds").list(),
                ]
            )
        )
        .sort("session")
        .to_pandas()
    )

    # PREDS = df_preds["preds"].tolist()
    # PREDS = [list(pred) for pred in PREDS]

    return df_preds

In [ ]:
sub = pd.read_csv("../data/01_raw/sample_submission.csv")

In [ ]:
%%time

model = RRNN()

model.load_state_dict(
    torch.load(
        f"{config.model_name}_epoch{0 + 1}.pth",
    )
)

model.to(device)

df_preds = inference_func(dl_merlin, model)
df_preds["session"] = list(map(str, df_preds["session"]))

dfs = list()
for session_type in ['clicks', 'carts', 'orders']:
    df_cur = df_preds[['session', "preds"]]
    df_cur.rename({"preds": 'labels', 'session': 'session_type'}, axis=1, inplace=True)
    df_cur['session_type']= df_cur['session_type'].progress_apply(lambda x: x + '_' + session_type)
    df_cur['labels']= df_cur['labels'].progress_apply(lambda x: ' '.join(str(aid) for aid in x))
    dfs.append(df_cur)

del df_cur
gc.collect()

pred_df = pd.concat(
    dfs
).reset_index(drop=True)

del dfs
gc.collect()

# pred_df = pl.from_pandas(pred_df)
# pred_df = pred_df.sort("session_type", reverse=False)

pred_df = sub[['session_type']].merge(pred_df, on=['session_type'])

pred_df.to_csv(f"../data/07_model_output/sub_last_week_ep1.csv", index=False)

In [ ]:
%%time

model = RRNN()

model.load_state_dict(
    torch.load(
        f"{config.model_name}_epoch{1 + 1}.pth",
    )
)

model.to(device)

df_preds = inference_func(dl_merlin, model)
df_preds["session"] = list(map(str, df_preds["session"]))

dfs = list()
for session_type in ['clicks', 'carts', 'orders']:
    df_cur = df_preds[['session', "preds"]]
    df_cur.rename({"preds": 'labels', 'session': 'session_type'}, axis=1, inplace=True)
    df_cur['session_type']= df_cur['session_type'].progress_apply(lambda x: x + '_' + session_type)
    df_cur['labels']= df_cur['labels'].progress_apply(lambda x: ' '.join(str(aid) for aid in x))
    dfs.append(df_cur)

del df_cur
gc.collect()

pred_df = pd.concat(
    dfs
).reset_index(drop=True)

del dfs
gc.collect()

# pred_df = pl.from_pandas(pred_df)
# pred_df = pred_df.sort("session_type", reverse=False)

pred_df = sub[['session_type']].merge(pred_df, on=['session_type'])

pred_df.to_csv(f"../data/07_model_output/sub_last_week_ep2.csv", index=False)

In [ ]:
%%time

model = RRNN()

model.load_state_dict(
    torch.load(
        f"{config.model_name}_epoch{4 + 1}.pth",
    )
)

model.to(device)

df_preds = inference_func(dl_merlin, model)
df_preds["session"] = list(map(str, df_preds["session"]))

dfs = list()
for session_type in ['clicks', 'carts', 'orders']:
    df_cur = df_preds[['session', "preds"]]
    df_cur.rename({"preds": 'labels', 'session': 'session_type'}, axis=1, inplace=True)
    df_cur['session_type']= df_cur['session_type'].progress_apply(lambda x: x + '_' + session_type)
    df_cur['labels']= df_cur['labels'].progress_apply(lambda x: ' '.join(str(aid) for aid in x))
    dfs.append(df_cur)

del df_cur
gc.collect()

pred_df = pd.concat(
    dfs
).reset_index(drop=True)

del dfs
gc.collect()

# pred_df = pl.from_pandas(pred_df)
# pred_df = pred_df.sort("session_type", reverse=False)

pred_df = sub[['session_type']].merge(pred_df, on=['session_type'])

pred_df.to_csv(f"../data/07_model_output/sub_last_week_ep5.csv", index=False)

In [ ]:
stop

In [ ]:
X_val[X_val["recall_row"] == 1].shape

In [ ]:
X_val_true = X_val[X_val["recall_row"] == 1]

In [ ]:
pad

In [ ]:
X_val_true.head()

In [ ]:
X_train[X_train["val"] == pad].shape[0] / X_train.shape[0]

In [ ]:
X_val_true.iloc[0]["preds"]

In [ ]:
stop
# loss_train: 12.190348625183105

# loss_train: 9.231197555859884
# loss_valid: 20.559587955474854
# metric_total: 0.000999000999000999

In [ ]:
params_sparsed = list()
params_not_sparsed = list()
for param in model.parameters():
    if param.grad.is_sparse:
        params_sparsed.append(param)
    else:
        params_not_sparsed.append(param)

In [ ]:
params_sparsed

In [ ]:
df_train.groupby(["n_week"]).count().sort("n_week", reverse=False)

In [ ]:
df_train_ex = df_train.filter(pl.col("user") <= 10000)

In [ ]:
%%time

dict_train = dict()
dict_answers = dict()
dict_val_len = dict()
dict_user = dict()
key = 1

for df_user in tqdm(df_train.groupby(["user"], maintain_order=True)):
    df_weeks_count = df_user.groupby(["n_week"]).count().sort("n_week", reverse=False)
    if len(df_weeks_count) <= 1:
        continue
    user = df_user["user"][0]
    for i in range(len(df_weeks_count["n_week"]) - 1):
        n_week_cur = df_weeks_count["n_week"][i]
        val_len = 34 - n_week_cur
        dict_train[key] = df_user.filter(pl.col("n_week") == n_week_cur)
        dict_answers[key] = df_user.filter(pl.col("n_week") > n_week_next)
        dict_val_len[key] = val_len
        dict_user[key] = user
        key += 1

In [ ]:
dict_train[1]

In [ ]:
dict_answers[1]

In [ ]:
df_user.filter(pl.col("n_week") == n_week_cur)

In [ ]:
for i in range(len(df_weeks_count["n_week"]) - 1):
    n_week_cur = df_weeks_count["n_week"][i]
    n_week_next = df_weeks_count["n_week"][i + 1]
    print(n_week_cur, n_week_next)

In [ ]:
df_user

In [ ]:
df_train_ex

In [ ]:
df_train.tail(10)

In [ ]:
df_test.head()

In [ ]:
utils.plot_user_activity(user=12899774, df=df_train)

In [ ]:
df_train["type"].value_counts()

In [ ]:
df_train["date"].min(), df_train["date"].max()

In [ ]:
df_test["date"].min(), df_test["date"].max()

In [ ]:
%%time

df_train.groupby(['dayofweek', 'type'])[['ts']].count()

In [ ]:
%%time

df_test.groupby(['dayofweek', 'type'])[['ts']].count()

In [ ]:
df_train_n_session = (df_train.groupby("user")[["n_session"]].max() + 1).reset_index()
df_test_n_session = (df_test.groupby("user")[["n_session"]].max() + 1).reset_index()

In [ ]:
df_train_n_session["n_session"].value_counts()

In [ ]:
df_test_n_session["n_session"].value_counts()

In [ ]:
def plot_n_interactions_per_dayofweek(df):
    try:
        df_name = df.name
    except AttributeError:
        df_name = None

    title = "n_interactions per dayofweek"
    if df_name is not None:
        title += f" in {df_name}"

    plt.figure(figsize=(12, 7))
    plt.title(title)
    plt.xlabel("dayofweek")
    plt.ylabel("n_interactions")

    dict_type_inter_offset = dict()
    dict_type_inter_offset["clicks"] = -0.2
    dict_type_inter_offset["carts"] = 0.0
    dict_type_inter_offset["orders"] = 0.2

    dict_type_inter_color = dict()
    dict_type_inter_color["clicks"] = "b"
    dict_type_inter_color["carts"] = "g"
    dict_type_inter_color["orders"] = "r"

    for (dayofweek, type_inter), df_cur in df.groupby(["dayofweek", "type"]):
        ys = df_cur["ts"].count()
        plt.bar(
            dayofweek + dict_type_inter_offset[type_inter],
            ys,
            width=0.2,
            color=dict_type_inter_color[type_inter],
            align="center",
            # label=feature,
            # marker=",",
            # linestyle="dashed",
            alpha=0.6,
        )

In [ ]:
plot_n_interactions_per_dayofweek(df_train)

In [ ]:
plot_n_interactions_per_dayofweek(df_test)

In [ ]:
def plot_user_activity(user, df):
    dict_type_inter_color = dict()
    dict_type_inter_color["clicks"] = "b"
    dict_type_inter_color["carts"] = "orange"
    dict_type_inter_color["orders"] = "r"

    offset = 0.5
    df_cur = df[df["user"] == user]
    df_cur["date_day"] = df_cur["date_day"].astype(str)

    if df.name == "df_train":
        x = pd.date_range("2022-08-01", "2022-08-28")
    elif df.name == "df_test":
        x = pd.date_range("2022-08-29", "2022-09-04")
    x = [str(d).split(" ")[0] for d in x]
    map_dates = {x[i]: i for i in range(len(x))}
    points_sundays = [point for point in range(len(x)) if (point + 1) % 7 == 0]

    plt.figure(figsize=(12, 7))
    plt.xticks(range(len(x)), x, rotation=90)
    plt.yticks(range(25))

    for point in range(25):
        plt.axhline(y=point, color="gray", linestyle="-", alpha=0.1)
    for point in range(len(x)):
        plt.axvline(x=point + offset, color="gray", linestyle="-", alpha=0.1)
        plt.axvline(x=point - offset, color="gray", linestyle="-", alpha=0.1)
    for point in points_sundays:
        plt.axvline(x=point + offset, color="r", linestyle="-", alpha=0.3)
        plt.axvline(x=point - offset, color="r", linestyle="-", alpha=0.3)
        plt.axvline(x=point - offset - 1, color="r", linestyle="-", alpha=0.3)

    for action, df_cur_action in df_cur.groupby("type"):
        xx = np.random.uniform(-0.15, 0.15, len(df_cur_action))
        yy = np.random.uniform(-0.15, 0.15, len(df_cur_action))
        plt.scatter(
            df_cur_action["date_day"].map(map_dates) + xx,
            df_cur_action["hour"] + yy,
            c=dict_type_inter_color[action],
            s=10,
            label=action,
        )
    plt.legend(
        loc="upper right", bbox_to_anchor=(1.1, 1.0), ncol=1, fancybox=True, shadow=True
    )

In [ ]:
plot_user_activity(user=2932269, df=df_train)

In [ ]:
plot_user_activity(user=12899780, df=df_test)